# 06 Plan C runs that stopped before epoch 30 (sensitivity analysis, added after the protocol)
In 23 plan C runs of the second wave (seeds 3 and 4; 21 with 10% and 2 with 25% of the training images) the protocol minimum
of 30 epochs was not in effect, and training stopped after 21 to 28 epochs. This notebook retrains exactly these 23 runs with
the protocol settings (minimum 30 epochs, patience 20), the same code, folds, seeds, factors and initial weights.
Outputs go to `runs_min30/` only; `runs/` is not touched. T4 GPU only. Two sessions can run at the same time
(set `SESSION = 'B'` in the second one). Do NOT run any smoke-test cell in these sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time
ROOT = '/content/drive/MyDrive/BUSI_QB'
CODE = os.path.join(ROOT, 'code'); PREP = os.path.join(ROOT, 'prepared'); RUNS = os.path.join(ROOT, 'runs')
RUNS_M = os.path.join(ROOT, 'runs_min30'); os.makedirs(RUNS_M, exist_ok=True)
sys.path.insert(0, os.path.join(CODE, 'src'))
SESSION = 'A'                                     # 'B' in a second session
import numpy as np, pandas as pd, tensorflow as tf
import qsim, qbus, runner
print(qbus.env_info())
assert 'T4' in str(qbus.env_info().get('gpu')), 'select a T4 GPU runtime'
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)
qbus.CFG.update(max_epochs=200, min_epochs=30, es_patience=20)
print({k: qbus.CFG[k] for k in ('max_epochs', 'min_epochs', 'es_patience', 'lr_patience')})
SELECTION = json.load(open(os.path.join(RUNS, 'plan_decision.json')))['selected']
AFFECTED = [('cl4', 0, 4, 0.1), ('cl4', 1, 4, 0.1), ('cl4', 2, 4, 0.1), ('cl4', 3, 3, 0.1), ('cl4', 4, 3, 0.1),
            ('cl4', 4, 4, 0.1), ('full', 0, 4, 0.1), ('full', 1, 4, 0.1), ('full', 2, 4, 0.1), ('full', 3, 4, 0.1),
            ('full', 4, 4, 0.1), ('q4', 0, 4, 0.1), ('q4', 1, 4, 0.1), ('q4', 1, 4, 0.25), ('q4', 3, 3, 0.1),
            ('q4', 3, 4, 0.1), ('q4', 4, 4, 0.25), ('tr4', 0, 4, 0.1), ('tr4', 1, 4, 0.1), ('tr4', 2, 4, 0.1),
            ('tr4', 3, 4, 0.1), ('tr4', 4, 3, 0.1), ('tr4', 4, 4, 0.1)]
todo = [dict(plan='C', model=m, fold=f, seed=s, frac=p) for m, f, s, p in AFFECTED]
done = glob.glob(os.path.join(RUNS_M, 'C', '*', 'summary.json'))
print(len(todo), 'runs in total,', len(done), 'done')

In [ ]:
# 2. RUNS ----------------------------------------------------------------------------------------------
root = os.path.join(RUNS_M, 'C'); os.makedirs(root, exist_ok=True)
for r in runner.ordered(todo, SESSION):
    rid = runner.run_id(r)
    if os.path.exists(os.path.join(root, rid, 'summary.json')):
        continue
    if runner.claim(root, rid, SESSION):
        assert qbus.CFG['min_epochs'] == 30 and qbus.CFG['max_epochs'] == 200, qbus.CFG
        print(time.strftime('%m-%d %H:%M'), 'start', rid)
        s = runner.one_run(r, PREP, root, selection=SELECTION, session=SESSION)
        old = json.load(open(os.path.join(RUNS, 'C', rid, 'summary.json')))
        print(f"   epochs {old['epochs']} -> {s['epochs']}, best epoch {old['best_epoch']} -> {s['best_epoch']}, "
              f"Dice {old['test_dice']:.4f} -> {s['test_dice']:.4f}, min {s['minutes']:.1f}")

In [ ]:
# 3. COLLECT (when all 23 runs are done) ------------------------------------------------------------------
dirs = sorted(glob.glob(os.path.join(RUNS_M, 'C', '*')))
res, cmp = [], []
for d in dirs:
    s = os.path.join(d, 'summary.json')
    if not os.path.exists(s):
        continue
    out = dict(summary=json.load(open(s))); rid = out['summary']['run_id']
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi'):
        f = os.path.join(d, nm + '.csv')
        if os.path.exists(f):
            t = pd.read_csv(f); t.insert(0, 'run_id', rid); out[nm] = t
    res.append(out)
    old = json.load(open(os.path.join(RUNS, 'C', rid, 'summary.json')))
    h_new = pd.read_csv(os.path.join(d, 'history.csv')); h_old = pd.read_csv(os.path.join(RUNS, 'C', rid, 'history.csv'))
    n = min(len(h_old), len(h_new))
    same = float(np.abs(h_new.val_loss.values[:n] - h_old.val_loss.values[:n]).max())
    cmp.append(dict(run_id=rid, epochs_old=old['epochs'], epochs_new=out['summary']['epochs'],
                    best_old=old['best_epoch'], best_new=out['summary']['best_epoch'],
                    dice_old=old['test_dice'], dice_new=out['summary']['test_dice'],
                    busbra_old=old.get('busbra_dice'), busbra_new=out['summary'].get('busbra_dice'),
                    breast_old=old.get('breast_dice'), breast_new=out['summary'].get('breast_dice'),
                    max_val_loss_diff_shared_epochs=same))
print(f'done {len(res)} / 23, running {len(glob.glob(os.path.join(RUNS_M, "C", "*.lock")))}')
if len(res) == 23:
    OUT = os.path.join(ROOT, 'analysis', 'min30'); os.makedirs(OUT, exist_ok=True)
    summ = pd.DataFrame([{k: (json.dumps(v) if isinstance(v, (dict, list)) else v) for k, v in r['summary'].items()} for r in res])
    summ.to_csv(os.path.join(OUT, 'summaries.csv'), index=False)
    for nm in ('test_busi', 'test_busbra', 'test_breast', 'normals_busi'):
        parts = [r[nm] for r in res if nm in r]
        if parts:
            t = pd.concat(parts, ignore_index=True)
            for c in t.columns:
                if t[c].dtype == float:
                    t[c] = t[c].round(5)
            t.to_csv(os.path.join(OUT, nm + '.csv.gz'), index=False, compression='gzip')
            print(nm, len(t), 'rows')
    c = pd.DataFrame(cmp); c.to_csv(os.path.join(OUT, 'comparison.csv'), index=False)
    print(c.round(4).to_string(index=False))
    print('changed best epoch:', int((c.best_old != c.best_new).sum()), '| max |Dice change|:', round(float((c.dice_new - c.dice_old).abs().max()), 4))
    print('MIN30 COLLECT OK')